# 13 Vetting Score

Composite Vetting Score (mimics real exoplanet vetting pipelines)
=====================================================================
Suggested notebook name: 13_vetting_score.ipynb (place in notebooks/)

WHY: Reproduces the named, transparent checks real pipelines (Kepler
Robovetter, TESS QLP) run -- independent of the ML model entirely.

IMPORTANT NAMING FIX: the old version's verdict field used the value
"likely_false_positive" -- now that the ML model has an actual class
literally called "false_positive", that name collision would be
confusing (the two are computed completely independently and can
disagree). Renamed the verdict column to rule_based_verdict and the
value to likely_false_positive_by_rules to make clear this is a
SEPARATE, independent check -- compare it against confidence_triage.csv's
ML predictions rather than treating them as the same thing.

Checks implemented:
    1. SNR check               -- signal strong enough to trust at all
    2. Odd-Even check          -- transit depth consistent between odd/even
                                   transits (large mismatch => eclipsing binary)
    3. Secondary eclipse check -- presence of a secondary dip (=> EB, not planet)
    4. Transit shape check     -- skewness close to flat-bottomed (planet-like)
                                   vs V-shaped (EB-like)

NOTE ON THRESHOLDS: defaults below are reasonable starting points, NOT
calibrated to your specific data. Run against labeled_features.csv first
(known planets/EBs/FPs) to sanity-check before trusting it on science stars.

In [1]:
import pandas as pd

ID_COL = "tic_id"
DATA_PATH = "../outputs/features.csv"
OUTPUT_PATH = "../outputs/vetting_report.csv"

SNR_MIN = 7.0
ODD_EVEN_MAX = 0.01
SECONDARY_DEPTH_MAX = 0.0005
SKEW_ABS_MAX = 1.0

df = pd.read_csv(DATA_PATH)
df["tic_id"] = df["tic_id"].astype(str).str.replace(".0", "", regex=False)

required = ["snr", "odd_even_diff", "secondary_depth", "transit_skewness"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns in {DATA_PATH}: {missing}\nAvailable: {list(df.columns)}")


def vet_row(row):
    checks = {
        "snr_check_pass": bool(row["snr"] >= SNR_MIN),
        "odd_even_check_pass": bool(row["odd_even_diff"] <= ODD_EVEN_MAX),
        "secondary_eclipse_check_pass": bool(row["secondary_depth"] <= SECONDARY_DEPTH_MAX),
        "transit_shape_check_pass": bool(abs(row["transit_skewness"]) <= SKEW_ABS_MAX),
    }
    score = sum(checks.values())
    checks["vetting_score"] = f"{score}/4"
    checks["rule_based_verdict"] = (
        "strong_candidate" if score == 4 else
        "weak_candidate" if score >= 2 else
        "likely_false_positive_by_rules"
    )
    return checks


vetting_results = df.apply(vet_row, axis=1, result_type="expand")
report = pd.concat([df[[ID_COL]], vetting_results], axis=1)
report.to_csv(OUTPUT_PATH, index=False)

print(report.to_string(index=False))
print(f"\nSaved -> {OUTPUT_PATH}")
print(f"\nVerdict counts:\n{report['rule_based_verdict'].value_counts().to_string()}")
print("\nThis is an INDEPENDENT rule-based check, separate from the ML model's")
print("predictions. Compare against confidence_triage.csv -- agreement between")
print("the two is your strongest evidence; disagreement is worth investigating.")

   tic_id  snr_check_pass  odd_even_check_pass  secondary_eclipse_check_pass  transit_shape_check_pass vetting_score rule_based_verdict
149603524           False                 True                          True                      True           3/4     weak_candidate
229742722           False                 True                          True                      True           3/4     weak_candidate
237201858           False                 True                          True                      True           3/4     weak_candidate
260647166            True                 True                          True                      True           4/4   strong_candidate
261136641           False                 True                          True                      True           3/4     weak_candidate
261136679            True                 True                          True                      True           4/4   strong_candidate
261136765           False                 True  